# NCKH 04 — SHAP Explainability for LightGBM

SHAP is used to explain model attribution for the selected multiclass
LightGBM. The global plot aggregates absolute contributions across
classes; the local plot focuses on the `CRITICAL` class for one
high-risk case. These values describe fitted-model behaviour and do
not establish causes or causal effects.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

ROOT = Path.cwd()
if not (ROOT / "reports" / "nckh_online_retail_multiclass").exists():
    ROOT = Path("..").resolve()
REPORT = ROOT / "reports" / "nckh_online_retail_multiclass"
SELECTED = REPORT / "selected"
FIG_DIR = REPORT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["CRITICAL", "HIGH_RISK", "MEDIUM_RISK", "LOW_RISK"]
CLASS_COLORS = {
    "CRITICAL": "#B42318",
    "HIGH_RISK": "#E76F00",
    "MEDIUM_RISK": "#176B87",
    "LOW_RISK": "#2A9D8F",
}
MODEL_LABELS = {
    "recent_zero_rule": "Recent-zero rule",
    "recent_quantile_rule": "Recent-sales quantile rule",
    "logistic_regression": "Logistic regression",
    "random_forest": "Random forest",
    "lightgbm": "LightGBM",
}
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

with open(SELECTED / "dataset_profile.json", encoding="utf-8") as handle:
    profile = json.load(handle)
with open(REPORT / "selected_config.json", encoding="utf-8") as handle:
    selected_config = json.load(handle)
with open(SELECTED / "metrics.json", encoding="utf-8") as handle:
    metrics = json.load(handle)

In [ ]:
importance = pd.read_csv(SELECTED / "shap_global_importance.csv")
local = json.loads((SELECTED / "shap_local_example.json").read_text(encoding="utf-8"))
DIAGNOSTICS = REPORT / "diagnostics"
shap_stability = pd.read_csv(DIAGNOSTICS / "shap_stability_by_cutoff.csv")
missed_critical = json.loads(
    (DIAGNOSTICS / "shap_local_missed_critical_case.json").read_text(encoding="utf-8")
)
importance.head(12)

## Figure 11 — Global SHAP importance

In [ ]:
top = importance.sort_values("mean_abs_shap_mean", ascending=True).tail(14)
fig, ax = plt.subplots(figsize=(9.2, 6.0))
bars = ax.barh(top["feature"], top["mean_abs_shap_mean"], color="#176B87")
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
ax.set_xlabel("Mean absolute SHAP value (mean over four classes)")
ax.set_title("Global LightGBM feature attribution")
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "11_shap_global_importance.png", bbox_inches="tight")
plt.show()

## Figure 12 — SHAP importance by class

In [ ]:
class_columns = [f"mean_abs_shap_{name}" for name in CLASS_NAMES]
heatmap = importance.set_index("feature")[class_columns].copy()
heatmap.columns = CLASS_NAMES
heatmap = heatmap.loc[importance.sort_values("mean_abs_shap_mean", ascending=False).head(14)["feature"]]
fig, ax = plt.subplots(figsize=(9.2, 6.5))
image = ax.imshow(heatmap.to_numpy(), aspect="auto", cmap="YlOrBr")
ax.set_yticks(range(len(heatmap.index)), heatmap.index)
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=25, ha="right")
ax.set_title("Mean absolute SHAP contribution by predicted class")
ax.set_xlabel("Explained class")
ax.set_ylabel("Feature")
fig.colorbar(image, ax=ax, label="Mean |SHAP|")
fig.tight_layout()
fig.savefig(FIG_DIR / "12_shap_by_class.png", bbox_inches="tight")
plt.show()

## Figure 13 — Local explanation for a CRITICAL prediction

In [ ]:
local_df = pd.DataFrame(local["top_feature_attributions"]).sort_values("absolute_shap")
colors = ["#B42318" if value > 0 else "#176B87" for value in local_df["shap_value"]]
fig, ax = plt.subplots(figsize=(9.2, 6.0))
ax.barh(local_df["feature"], local_df["shap_value"], color=colors)
ax.axvline(0, color="#212529", linewidth=0.8)
ax.set_xlabel("SHAP contribution to CRITICAL raw score")
ax.set_title(
    f"Local attribution: SKU {local['case']['sku']} at {local['case']['cutoff_date']}"
)
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "13_shap_local_critical_case.png", bbox_inches="tight")
plt.show()
print("Case:", local["case"])
print(local["interpretation_boundary"])

**Interpretation boundary.** Positive SHAP values increase the model's
raw score for the explained class and negative values decrease it.
Correlated rolling features may share attribution. The chart should be
described as a transparent account of model scoring, not as evidence
that a feature caused a product to become slow-moving.

## Figure 18 - SHAP-ranking stability across test cutoffs

In [ ]:
top_features = importance.nlargest(6, "mean_abs_shap_mean")["feature"].tolist()
stability_top = shap_stability.loc[shap_stability["feature"].isin(top_features)].copy()
stability_top["cutoff_date"] = pd.to_datetime(stability_top["cutoff_date"])
fig, ax = plt.subplots(figsize=(9.4, 5.2))
for feature in top_features:
    values = stability_top.loc[stability_top["feature"].eq(feature)]
    ax.plot(
        values["cutoff_date"], values["mean_abs_shap_mean"],
        marker="o", linewidth=2, label=feature,
    )
ax.set_xlabel("Test decision cutoff")
ax.set_ylabel("Mean |SHAP| over four classes")
ax.set_title("Stability of the leading SHAP features across test cutoffs")
ax.legend(frameon=False, fontsize=8, ncol=2)
ax.grid(alpha=0.2)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIG_DIR / "18_shap_stability.png", bbox_inches="tight")
plt.show()

## Figure 19 - Local SHAP explanation of a missed CRITICAL case

In [ ]:
missed_df = pd.DataFrame(missed_critical["top_feature_attributions"]).sort_values(
    "shap_value_for_CRITICAL_raw_score"
)
colors = [
    "#B42318" if value > 0 else "#176B87"
    for value in missed_df["shap_value_for_CRITICAL_raw_score"]
]
fig, ax = plt.subplots(figsize=(9.4, 6.0))
ax.barh(
    missed_df["feature"],
    missed_df["shap_value_for_CRITICAL_raw_score"],
    color=colors,
)
ax.axvline(0, color="#212529", linewidth=0.8)
ax.set_xlabel("SHAP contribution to CRITICAL raw score")
ax.set_title(
    "Why a CRITICAL case was predicted LOW_RISK: "
    f"SKU {missed_critical['case']['sku']} at "
    f"{pd.Timestamp(missed_critical['case']['cutoff_date']).date()}"
)
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "19_shap_missed_critical_case.png", bbox_inches="tight")
plt.show()
print("Case:", missed_critical["case"])
print(missed_critical["interpretation_boundary"])